In [ ]:
# Cell 1 — Setup + data load (#027 intraday FX stat-arb)
import pandas as pd
from backtest.data import DataPanel

prices = pd.read_parquet("data/fx_intraday_m15.parquet").sort_index()
prices = prices[~prices.index.duplicated(keep="first")]   # engine requires unique, monotonic index

panel = DataPanel(prices, check_outliers=True)

print(f"Bars:   {len(panel.dates):,}")
print(f"Assets: {len(panel.assets_all)} -> {list(panel.assets_all)}")
print(f"Range:  {panel.dates[0]} -> {panel.dates[-1]}   tz={prices.index.tz}")
print(f"NaN fraction in prices: {prices.isna().mean().mean():.4%}")
print(f"index monotonic increasing: {prices.index.is_monotonic_increasing}, unique: {prices.index.is_unique}")
print("\nprices.tail(3):")
print(prices.tail(3).to_string())


In [ ]:
# Cell 2 — Strategy class: discrete PCA-residual s-score stat-arb (#027)
import numpy as np
import pandas as pd
from backtest.strategy import Strategy

_GAP = pd.Timedelta("20min")   # consecutive-bar gap threshold (mask weekend/holiday returns)


class StatArbResidual(Strategy):
    """Intraday cross-sectional stat-arb. Each bar: strip top-m PCA factors from standardized
    returns -> idiosyncratic residual; OU s-score on the trailing-L cumulative residual;
    DISCRETE bands per pair (open |s|>s_in, hold, close |s|<s_out), fixed 1/N sizing.
    Market-neutral, intraday (no overnight). PCA re-estimated once per calendar day (cached on self)."""

    rebalance_frequency = "daily"          # = every bar in `dates` (base.py:67-68); here every M15 bar

    def __init__(self, pca_win=288, m_factors=3, L=32, vol_win=96, s_in=2.0, s_out=0.5):
        self.pca_win = pca_win             # PCA estimation window (bars; 288 = 3 trading days)
        self.m_factors = m_factors         # common factors removed
        self.L = L                         # s-score / OU window (bars; 32*15min = 8h)
        self.vol_win = vol_win             # return-standardization window (bars; 96 = 1 day)
        self.s_in = s_in                   # entry threshold
        self.s_out = s_out                 # exit threshold
        # state (persists across generate_weights within one run; per §11)
        self._pos = None                   # current discrete position per asset in {-1,0,+1}
        self._day = None                   # calendar day of the cached PCA loadings
        self._V = None                     # cached top-m eigenvectors

    def __repr__(self):
        return (f"StatArbResidual(pca_win={self.pca_win},m={self.m_factors},L={self.L},"
                f"vol_win={self.vol_win},s_in={self.s_in},s_out={self.s_out})")

    def apply_risk(self, proposed, state, data):
        return proposed                    # identity — faithful signal replay (risk = optional Stage 5)

    def _sscore(self, px, t):
        """PIT s-score per pair at t, from prices up to t. Returns pd.Series or None if warming up."""
        need = self.pca_win + self.vol_win + 10
        if len(px) < need:
            return None
        px = px.iloc[-need:]
        rets = np.log(px).diff()
        rets[(px.index.to_series().diff() > _GAP).values] = np.nan       # mask gap returns
        vol = rets.rolling(self.vol_win, min_periods=self.vol_win // 2).std()
        z = (rets / vol).clip(-8, 8)
        # re-estimate PCA loadings once per calendar day, cache on self
        day = t.normalize()
        if self._day != day or self._V is None:
            Zw = z.iloc[-self.pca_win:].dropna()
            if len(Zw) < len(px.columns) * 5:
                return None
            C = np.nan_to_num(np.corrcoef(Zw.values.T))
            evals, evecs = np.linalg.eigh(C)
            self._V = evecs[:, np.argsort(evals)[::-1][:self.m_factors]]
            self._day = day
        P = self._V @ self._V.T                                          # factor projection
        Z = z.iloc[-2 * self.L:].fillna(0.0).values                      # last 2L bars
        R = Z - Z @ P.T                                                   # factor-neutral residual
        X = pd.DataFrame(R, columns=px.columns).rolling(self.L).sum().iloc[-self.L:]
        s = -(X.iloc[-1] - X.mean()) / X.std().replace(0, np.nan)        # OU s-score at t
        return s

    def generate_weights(self, data, t):
        px = data.prices
        assets = data.assets
        if self._pos is None:
            self._pos = pd.Series(0.0, index=px.columns)
        s = self._sscore(px, t)
        if s is None:
            return pd.Series(0.0, index=assets)                          # warmup -> flat
        # discrete state machine per pair (long when residual cheap, i.e. s > s_in)
        for a in px.columns:
            v = s.get(a, np.nan)
            if not np.isfinite(v):
                continue                                                 # hold through missing signal
            p = self._pos[a]
            if p == 0.0:
                if v > self.s_in:      self._pos[a] = 1.0
                elif v < -self.s_in:   self._pos[a] = -1.0
            elif p == 1.0 and v < self.s_out:      self._pos[a] = 0.0
            elif p == -1.0 and v > -self.s_out:    self._pos[a] = 0.0
        N = len(px.columns)                                              # fixed 1/N sizing per pair
        return (self._pos / N).reindex(assets).fillna(0.0)


strat = StatArbResidual()
print(strat)
rb = strat.rebalance_dates(panel.dates)
print(f"rebalance bars: {len(rb):,} of {len(panel.dates):,} panel bars  (every-bar: {len(rb) == len(panel.dates)})")
print(f"approx rebalances/year: {len(rb) / (len(panel.dates) / (96*252)):,.0f}  (M15: 96*252 bars/yr)")
print(f"warmup bars before first signal: ~{strat.pca_win + strat.vol_win + 10}")


In [ ]:
# Cell 3 — Data revision: mid prices + per-bar spread feature (#027)
import pandas as pd
from backtest.data import DataPanel, FieldSpec

bid = pd.read_parquet("data/fx_intraday_m15.parquet").sort_index()
ask = pd.read_parquet("data/fx_intraday_ask_m15.parquet").sort_index()
bid = bid[~bid.index.duplicated(keep="first")]
ask = ask[~ask.index.duplicated(keep="first")].reindex(index=bid.index, columns=bid.columns)

mid = (bid + ask) / 2.0                                  # mark-to-market / signal prices
half_bp = (ask - bid) / 2.0 / mid * 1e4                  # per-bar half-spread, in bp

panel = DataPanel(
    prices=mid,
    features={"spread": half_bp},
    specs={"spread": FieldSpec(lag=0, missing="ffill", max_staleness=4)},
    check_outliers=True,
)

print("available_fields:", panel.available_fields())
print(f"prices(mid) {mid.shape} | spread feature {half_bp.shape} | "
      f"spread NaN frac (pre-ffill) {half_bp.isna().mean().mean():.4%}")
print("\nper-pair half-spread (bp) percentiles:")
q = half_bp.quantile([.5, .9, .95, .99]).T
q.columns = ["p50", "p90", "p95", "p99"]
print(q.round(3).to_string())


In [ ]:
# Cell 4 — Strategy revision: add the spread gate (open only) (#027)
import numpy as np
import pandas as pd
from backtest.strategy import Strategy

_GAP = pd.Timedelta("20min")


class StatArbResidual(Strategy):
    """Discrete PCA-residual s-score stat-arb with a SPREAD GATE: open a position only when the
    pair's half-spread at t is <= spread_cap; closes are never gated. Market-neutral, intraday.
    PCA re-estimated once per calendar day (cached on self)."""

    rebalance_frequency = "daily"          # = every bar (base.py:67-68) -> every M15 bar

    def __init__(self, pca_win=288, m_factors=3, L=32, vol_win=96, s_in=2.0, s_out=0.5, spread_cap=0.20):
        self.pca_win = pca_win
        self.m_factors = m_factors
        self.L = L
        self.vol_win = vol_win
        self.s_in = s_in
        self.s_out = s_out
        self.spread_cap = spread_cap       # bp; skip an OPEN if half-spread(t) > cap
        self._pos = None
        self._day = None
        self._V = None

    def __repr__(self):
        return (f"StatArbResidual(pca_win={self.pca_win},m={self.m_factors},L={self.L},"
                f"vol_win={self.vol_win},s_in={self.s_in},s_out={self.s_out},spread_cap={self.spread_cap})")

    def required_data(self):
        return {"prices": None, "spread": None}

    def apply_risk(self, proposed, state, data):
        return proposed                    # identity — risk = optional Stage 5

    def _sscore(self, px, t):
        need = self.pca_win + self.vol_win + 10
        if len(px) < need:
            return None
        px = px.iloc[-need:]
        rets = np.log(px).diff()
        rets[(px.index.to_series().diff() > _GAP).values] = np.nan
        vol = rets.rolling(self.vol_win, min_periods=self.vol_win // 2).std()
        z = (rets / vol).clip(-8, 8)
        day = t.normalize()
        if self._day != day or self._V is None:
            Zw = z.iloc[-self.pca_win:].dropna()
            if len(Zw) < len(px.columns) * 5:
                return None
            C = np.nan_to_num(np.corrcoef(Zw.values.T))
            evals, evecs = np.linalg.eigh(C)
            self._V = evecs[:, np.argsort(evals)[::-1][:self.m_factors]]
            self._day = day
        P = self._V @ self._V.T
        Z = z.iloc[-2 * self.L:].fillna(0.0).values
        R = Z - Z @ P.T
        X = pd.DataFrame(R, columns=px.columns).rolling(self.L).sum().iloc[-self.L:]
        return -(X.iloc[-1] - X.mean()) / X.std().replace(0, np.nan)

    def generate_weights(self, data, t):
        px = data.prices
        assets = data.assets
        if self._pos is None:
            self._pos = pd.Series(0.0, index=px.columns)
        s = self._sscore(px, t)
        if s is None:
            return pd.Series(0.0, index=assets)
        sp = data.feature("spread").iloc[-1]              # per-pair half-spread(bp) at t
        for a in px.columns:
            v = s.get(a, np.nan)
            if not np.isfinite(v):
                continue                                  # hold through missing signal
            p = self._pos[a]
            if p == 0.0:                                  # OPEN — spread-gated
                spread_a = sp.get(a, np.nan)
                if np.isfinite(spread_a) and spread_a <= self.spread_cap:
                    if v > self.s_in:      self._pos[a] = 1.0
                    elif v < -self.s_in:   self._pos[a] = -1.0
            elif p == 1.0 and v < self.s_out:      self._pos[a] = 0.0    # CLOSE — never gated
            elif p == -1.0 and v > -self.s_out:    self._pos[a] = 0.0
        N = len(px.columns)
        return (self._pos / N).reindex(assets).fillna(0.0)


strat = StatArbResidual(spread_cap=0.20)
print(strat)
print("\ngate pass-rate (frac of bars with half-spread <= cap), by pair:")
print((half_bp <= strat.spread_cap).mean().sort_values(ascending=False).round(3).to_string())


In [ ]:
# Cell 5 — Strategy revision: force flat across the FX rollover (swap-free) (#027)
import numpy as np
import pandas as pd
from backtest.strategy import Strategy

_GAP = pd.Timedelta("20min")


class StatArbResidual(Strategy):
    """Discrete PCA-residual s-score stat-arb. Spread-gated opens (<= spread_cap); closes never gated.
    FLAT across the 17:00-NY FX rollover (swap-free). Market-neutral, intraday. PCA cached per day."""

    rebalance_frequency = "daily"          # every bar -> every M15 bar

    def __init__(self, pca_win=288, m_factors=3, L=32, vol_win=96, s_in=2.0, s_out=0.5, spread_cap=0.20):
        self.pca_win = pca_win
        self.m_factors = m_factors
        self.L = L
        self.vol_win = vol_win
        self.s_in = s_in
        self.s_out = s_out
        self.spread_cap = spread_cap
        self._pos = None
        self._day = None
        self._V = None

    def __repr__(self):
        return (f"StatArbResidual(pca_win={self.pca_win},m={self.m_factors},L={self.L},"
                f"vol_win={self.vol_win},s_in={self.s_in},s_out={self.s_out},spread_cap={self.spread_cap})")

    def required_data(self):
        return {"prices": None, "spread": None}

    def apply_risk(self, proposed, state, data):
        return proposed

    def _sscore(self, px, t):
        need = self.pca_win + self.vol_win + 10
        if len(px) < need:
            return None
        px = px.iloc[-need:]
        rets = np.log(px).diff()
        rets[(px.index.to_series().diff() > _GAP).values] = np.nan
        vol = rets.rolling(self.vol_win, min_periods=self.vol_win // 2).std()
        z = (rets / vol).clip(-8, 8)
        day = t.normalize()
        if self._day != day or self._V is None:
            Zw = z.iloc[-self.pca_win:].dropna()
            if len(Zw) < len(px.columns) * 5:
                return None
            C = np.nan_to_num(np.corrcoef(Zw.values.T))
            evals, evecs = np.linalg.eigh(C)
            self._V = evecs[:, np.argsort(evals)[::-1][:self.m_factors]]
            self._day = day
        P = self._V @ self._V.T
        Z = z.iloc[-2 * self.L:].fillna(0.0).values
        R = Z - Z @ P.T
        X = pd.DataFrame(R, columns=px.columns).rolling(self.L).sum().iloc[-self.L:]
        return -(X.iloc[-1] - X.mean()) / X.std().replace(0, np.nan)

    def generate_weights(self, data, t):
        px = data.prices
        assets = data.assets
        if self._pos is None:
            self._pos = pd.Series(0.0, index=px.columns)
        # FLAT across the 17:00-NY FX rollover snapshot (swap-free)
        t_ny = t.tz_convert("America/New_York")
        if (t_ny.hour == 16 and t_ny.minute >= 45) or (t_ny.hour == 17 and t_ny.minute < 15):
            self._pos[:] = 0.0
            return pd.Series(0.0, index=assets)
        s = self._sscore(px, t)
        if s is None:
            return pd.Series(0.0, index=assets)
        sp = data.feature("spread").iloc[-1]
        for a in px.columns:
            v = s.get(a, np.nan)
            if not np.isfinite(v):
                continue
            p = self._pos[a]
            if p == 0.0:                                  # OPEN — spread-gated
                spread_a = sp.get(a, np.nan)
                if np.isfinite(spread_a) and spread_a <= self.spread_cap:
                    if v > self.s_in:      self._pos[a] = 1.0
                    elif v < -self.s_in:   self._pos[a] = -1.0
            elif p == 1.0 and v < self.s_out:      self._pos[a] = 0.0    # CLOSE — never gated
            elif p == -1.0 and v > -self.s_out:    self._pos[a] = 0.0
        N = len(px.columns)
        return (self._pos / N).reindex(assets).fillna(0.0)


strat = StatArbResidual(spread_cap=0.20)
print(strat)
ny = panel.dates.tz_convert("America/New_York")
flat_mask = ((ny.hour == 16) & (ny.minute >= 45)) | ((ny.hour == 17) & (ny.minute < 15))
print(f"flatten-window bars: {flat_mask.sum():,} ({flat_mask.sum()/ (len(panel.dates)/(96*252)):.0f}/yr, ~{flat_mask.mean()*96:.1f}/day)")


In [ ]:
# Cell 6 — Costs: custom time-varying spread cost model (#027)
from backtest.costs import CompositeCostModel
from backtest.costs.base import CostModel


class TimeVaryingSpread(CostModel):
    """Charge the REAL per-bar half-spread on each executed trade: |trades| * half_bp(t) / 1e4.
    Reads the 'spread' feature (per-pair half-spread in bp) PIT via the DataView. holding_cost=0
    (swap-free by the rollover flatten). Time-varying analogue of the built-in Spread (components.py:36-46)."""

    def trade_cost(self, trades, view):
        half_bp = view.feature("spread").iloc[-1].reindex(trades.index).fillna(0.0)
        return float((trades.abs() * half_bp / 1e4).sum())


costs = CompositeCostModel([TimeVaryingSpread()])
print("Cost components:", [type(m).__name__ for m in costs.models])
print("panel has 'spread' field:", panel.has_field("spread"))
print("panel has 'volume' field:", panel.has_field("volume"), "(no MarketImpact/LiquidityCap -> fine)")


In [ ]:
# Cell 7 — Single-path engine run (#027)
from backtest.engine import Engine

WARMUP_BARS = 500
engine = Engine(panel, strat, costs=costs, initial_capital=1_000_000)
result = engine.run(start=panel.dates[WARMUP_BARS])

print(f"Bars:                {len(result.equity_curve):,}")
print(f"Rebalances:          {result.metadata['n_rebalances']:,}")
print(f"Date range:          {result.metadata['start'].date()} -> {result.metadata['end'].date()}")
print(f"Initial:             ${result.metadata['initial_capital']:>14,.0f}")
print(f"Final equity:        ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:        {(result.equity_curve.iloc[-1]/result.metadata['initial_capital']-1):+.2%}")
print(f"Total costs:         ${result.costs.sum():>14,.0f}")
print(f"Max position weight: {result.weights.abs().max().max():.3%}")
print(f"Max gross exposure:  {result.weights.abs().sum(axis=1).max():.3%}")
print("DONE")


In [ ]:
# Cell 8 — TRIAL 2 (best-net shot): high conviction + tight gate + swap-free (#027)
from backtest.engine import Engine

strat2 = StatArbResidual(s_in=3.0, s_out=0.75, spread_cap=0.18)
res2 = Engine(panel, strat2, costs=costs, initial_capital=1_000_000).run(start=panel.dates[500])

r = res2.returns.dropna()
gross_r = (res2.gross_pnl / res2.equity_curve.shift(1)).dropna()
ann = (96 * 252) ** 0.5
def sh(x): return x.mean() / x.std() * ann if x.std() > 0 else 0.0

print(strat2)
print(f"NET   total {res2.equity_curve.iloc[-1]/1e6 - 1:+.2%}  ann {r.mean()*96*252:+.2%}  Sharpe {sh(r):+.2f}")
print(f"GROSS total {res2.gross_pnl.sum()/1e6:+.2%}  ann {gross_r.mean()*96*252:+.2%}  Sharpe {sh(gross_r):+.2f}")
print(f"costs {res2.costs.sum()/1e6:.2%}  |  max gross exposure {res2.weights.abs().sum(axis=1).max():.2%}")
yr = r.groupby(r.index.year).apply(lambda x: sh(x))
print("per-year NET Sharpe:", {int(y): round(float(v), 2) for y, v in yr.items()})
print("DONE")
